<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.it/cap06/cap06.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

## 💻 Parte Pratica con Esercizi di Programmazione

Gli esercizi di programmazione (EP) di questa sezione completano i concetti presentati lungo il Capitolo 6 attraverso l’implementazione di algoritmi legati all’ispezione industriale e all’analisi dei documenti. L’obiettivo è consolidare i fondamenti studiati, riproducendo, in scala ridotta, le fasi di una *pipeline* tipica della Visione Artificiale.

A differenza dei capitoli precedenti, i cui esercizi enfatizzavano operazioni più direttamente legate ai dati immagine, gli EP di questo capitolo si concentrano sulle **grandezze intermedie** prodotte durante l’elaborazione, come aree, perimetri, circolarità, angoli di rette, gradi di riempimento di bolle, mappe di varianza e mappe di differenza. Questo approccio permette di comprendere e validare ogni fase della *pipeline* in modo indipendente, senza dipendere da librerie specializzate per l’acquisizione di immagini, la rilevazione di marcatori o la decodifica di codici — ad eccezione dell’esercizio di chiusura del capitolo (EP06_08), che introduce intenzionalmente l’uso di OpenCV per la segmentazione e la decodifica reale di un *QRCode*, chiudendo il ciclo tra i concetti teorici e gli strumenti impiegati nella pratica.

Gli esercizi seguono la stessa sequenza concettuale del capitolo, in ordine crescente di complessità. Inizialmente vengono affrontate le metriche di valutazione della segmentazione, utilizzate per quantificare la qualità delle maschere binarie. Successivamente, si studiano criteri geometrici per la selezione dei marcatori, la classificazione delle marcature nei moduli e la stima dell’inclinazione dei documenti tramite la Trasformata di Hough. Nella parte finale, gli esercizi esplorano la normalizzazione dell’illuminazione, il rilevamento dei difetti tramite analisi della trama e l’integrazione tra registrazione geometrica e sottrazione di immagini in una *pipeline* semplificata di ispezione industriale.

Ogni esercizio rappresenta una fase isolata di un sistema reale di Visione Artificiale, consentendo di validare singolarmente concetti che, nelle applicazioni industriali, vengono combinati in un’unica *pipeline* di ispezione.

### 🗺️ Legenda della Difficoltà

| Livello | Significato | EP |
|:---:|---|---|
| 🟢 | Molto facile / facile — implementazione di un singolo concetto o algoritmo semplice | EP06_01, EP06_02 |
| 🟡 | Facile–medio — gestione di più casi o utilizzo di criteri statistici semplici | EP06_03, EP06_04 |
| 🟠 | Medio — elaborazione matriciale punto a punto | EP06_05 |
| 🔴 | Difficile — elaborazione matriciale con operazioni di vicinato (finestra scorrevole) | EP06_06 |
| 🟣 | Molto difficile — integrazione di più fasi di una *pipeline* di Visione Artificiale | EP06_07 |
| ⚫ | Speciale — uso di una libreria specializzata (`cv2`) per la segmentazione geometrica e la decodifica reale di codici a barre/QRCode | EP06_08 |

> ### ❗ Linee guida per la Risoluzione degli Esercizi di Programmazione
>
> Salvo diversa indicazione, tutti gli esercizi utilizzano la convenzione di coordinate matriciali `[riga][colonna]`, con origine in $(0,0)$ nell’angolo superiore sinistro dell’immagine.
>
> Quando è necessario un arrotondamento numerico, si deve utilizzare l’arrotondamento standard al numero intero più vicino (*round half away from zero*, con `np.floor(img + 0.5)`). I confronti con soglie (ad esempio, circolarità, varianza, differenza di intensità o grado di riempimento) devono essere considerati **stretti** (`>`), salvo che il testo non specifichi esplicitamente un altro criterio.
>
> Ogni esercizio è stato ideato per enfatizzare un concetto specifico presentato nel capitolo. Si raccomanda di implementare inizialmente la soluzione in modo diretto e, solo dopo la sua validazione, cercare alternative più efficienti o più generali.

### 🎯 Obiettivo di questo Quaderno

Questo quaderno è stato realizzato per supportare lo sviluppo, la validazione e i test delle soluzioni degli **Esercizi di Programmazione (EP)** in un ambiente interattivo, come Google Colab o Jupyter Notebook. Dopo aver verificato il funzionamento dell'implementazione con i casi di test presentati, il codice può essere inviato a Moodle per la valutazione ufficiale.

#### *Download*

Esegui la cella seguente per ottenere i file `morph.py` e `testsuite.py`, utilizzati dagli esercizi di questo capitolo.

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Esecuzione dei Test

Dopo aver implementato la soluzione, esegui `TestSuite("EP06_01.estensione").run()` in una nuova cella, sostituendo `estensione` con il linguaggio utilizzato (`.py`, `.java`, `.c`, `.cpp`, `.js` o `.r`). Il sistema recupera automaticamente i casi di test dal repository del corso, esegue il programma e presenta il risultato della valutazione.

In Python, è anche possibile testare la soluzione direttamente da una *stringa*, senza la necessità di salvare il codice in un file. A tale scopo, memorizza il programma in una variabile e utilizza il metodo `run_code`:

```python
codigo = """
# ... il tuo codice qui ...
"""

TestSuite("EP06_01").run_code(codigo)
```

### EP06_01 🟢 Valutazione della Segmentazione tramite IoU (*Intersection over Union*)

Nel corso di questo capitolo, diverse fasi del *pipeline* producono **maschere binarie**, come nella segmentazione di documenti, nella localizzazione di *QRCodes* e nel rilevamento di difetti. Per valutare oggettivamente la qualità di queste segmentazioni, è necessario confrontarle con una maschera di riferimento (*ground truth*).

Una delle metriche più utilizzate a questo scopo è la **IoU** (*Intersection over Union*, o Intersezione su Unione), definita come il rapporto tra l'area di intersezione e l'area di unione di due maschere binarie. Quanto maggiore è il valore della IoU, tanto maggiore è la concordanza tra la segmentazione prodotta dall'algoritmo e il riferimento.

#### 📋 Linee Guida di Implementazione

1. **Dimensioni:** Leggere gli interi $L$ (numero di righe) e $C$ (numero di colonne).
2. **Maschera di riferimento:** Leggere gli $L \times C$ elementi binari (0 o 1) della matrice `ref`.
3. **Maschera predetta:** Leggere gli $L \times C$ elementi binari (0 o 1) della matrice `pred`.
4. **Intersezione:** Contare il numero di posizioni $(i,j)$ per le quali `ref[i][j] = 1` e `pred[i][j] = 1`.
5. **Unione:** Contare il numero di posizioni $(i,j)$ per le quali `ref[i][j] = 1` o `pred[i][j] = 1`.
6. **Caso degenere:** Se l'unione è uguale a $0$, definire $\mathrm{IoU}=1{,}0$, poiché entrambe le maschere sono vuote.
7. **Calcolo:** Se l'unione è maggiore di zero, calcolare

$$
\mathrm{IoU}=
\frac{|\mathrm{Intersezione}|}
{|\mathrm{Unione}|}.
$$

8. **Classificazione:** Determinare la classificazione qualitativa utilizzando il valore di IoU **prima** dell'arrotondamento.
9. **Arrotondamento:** Visualizzare la IoU con quattro cifre decimali.
10. **Output:** Stampare, in quest'ordine, l'intersezione, l'unione, la IoU e la classificazione.

#### 📌 Vincoli Computazionali

- Se l'unione è uguale a $0$, non deve essere eseguita la divisione; la IoU deve essere definita come $1{,}0$.
- Le fasce di classificazione utilizzano confronti non stretti ($\geq$).
- La classificazione deve essere eseguita utilizzando il valore della IoU a piena precisione, prima dell'arrotondamento per la visualizzazione.

#### 🧠 Fondamento Teorico

La IoU è definita da

$$
\mathrm{IoU}=
\frac{|R\cap P|}
{|R\cup P|},
$$

dove:

- $R$ rappresenta l'insieme dei pixel appartenenti alla maschera di riferimento;
- $P$ rappresenta l'insieme dei pixel appartenenti alla maschera predetta;
- $|R\cap P|$ corrisponde al numero di pixel appartenenti simultaneamente a entrambe le maschere;
- $|R\cup P|$ corrisponde al numero di pixel appartenenti ad almeno una delle maschere.

| Fascia di IoU | Classificazione | Interpretazione |
|---|---|---|
| $\mathrm{IoU}\geq0{,}90$ | `ECCELLENTE` | Concordanza molto elevata tra le maschere. |
| $0{,}70\leq\mathrm{IoU}<0{,}90$ | `BUONO` | Piccole differenze tra le maschere. |
| $0{,}50\leq\mathrm{IoU}<0{,}70$ | `ACCETTABILE` | Concordanza parziale tra le maschere. |
| $\mathrm{IoU}<0{,}50$ | `SCARSO` | Bassa concordanza tra le maschere. |

La IoU dipende solo dalla sovrapposizione tra le maschere e, pertanto, è indipendente dalla dimensione dell'immagine.

#### 📦 Specifica di Input e Output (VPL)

**Input:**

- Riga 1: intero $L$.
- Riga 2: intero $C$.
- Prossime $L$ righe: elementi binari (0 o 1) della matrice `ref`.
- Prossime $L$ righe: elementi binari (0 o 1) della matrice `pred`.

**Output:**

- Riga 1: `Intersezione: X`
- Riga 2: `Unione: Y`
- Riga 3: `IoU: Z`
- Riga 4: `Classificazione: NOME`

Il valore di `IoU` deve essere stampato con quattro cifre decimali.

#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 2<br>2<br>1 1<br>0 0<br>1 0<br>0 0 | Intersezione: 1<br>Unione: 2<br>IoU: 0.5000<br>Classificazione: ACCETTABILE | La metà della regione di riferimento è stata segmentata correttamente. |
| 2<br>2<br>0 0<br>0 0<br>0 0<br>0 0 | Intersezione: 0<br>Unione: 0<br>IoU: 1.0000<br>Classificazione: ECCELLENTE | Entrambe le maschere sono vuote; per convenzione, $\mathrm{IoU}=1{,}0$. |

In [ ]:
from IPython.display import HTML

HTML("""
<div id="sim-ep0601" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0601 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0601 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0601 button:hover { background: #e8dfcf; }
  #sim-ep0601 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0601_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0601_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0601_grid_ctrls { display: grid; grid-template-columns: repeat(auto-fit, minmax(140px, 1fr)); gap: 12px; }
  .sim-ep0601_px { width: 24px; height: 24px; border: 1px solid #e4dcc8; box-sizing: border-border; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP06_01: IoU (Intersection over Union)</span>
  <span class="sim-ep0601_pill">IoU = |A &cap; B| / |A &cup; B|</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Controles -->
  <div class="sim-ep0601_panel" style="margin-bottom:14px;">
    <div class="sim-ep0601_grid_ctrls">
      
      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Spostamento H (&Delta;x)</label>
          <span id="sim-ep0601_vdx" style="font-family:monospace; font-weight:700; color:#26241d;">0</span>
        </div>
        <input id="sim-ep0601_dx" type="range" min="-3" max="3" step="1" value="0">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Spostamento V (&Delta;y)</label>
          <span id="sim-ep0601_vdy" style="font-family:monospace; font-weight:700; color:#26241d;">0</span>
        </div>
        <input id="sim-ep0601_dy" type="range" min="-3" max="3" step="1" value="0">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Lato del Quadrato</label>
          <span id="sim-ep0601_vsz" style="font-family:monospace; font-weight:700; color:#26241d;">6</span>
        </div>
        <input id="sim-ep0601_sz" type="range" min="2" max="8" step="1" value="6">
      </div>

    </div>

    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:10px; text-align:center;">
      Sposta e ridimensiona la maschera prevista per valutare l'allineamento.
    </div>
  </div>

  <!-- Exibição das Máscaras 10x10 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap:12px; margin-bottom:14px;">
    
    <div class="sim-ep0601_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        Riferimento (A)
      </div>
      <div id="sim-ep0601_ref" style="display:grid; grid-template-columns:repeat(10, 24px); gap:2px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0601_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        Prevista (B)
      </div>
      <div id="sim-ep0601_pred" style="display:grid; grid-template-columns:repeat(10, 24px); gap:2px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0601_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        Sovrapposizione (A &cap; B)
      </div>
      <div id="sim-ep0601_mix" style="display:grid; grid-template-columns:repeat(10, 24px); gap:2px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0601_dbg" class="sim-ep0601_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep01(root){
    if (!root || root.dataset.sim06Ep01Init) return;
    root.dataset.sim06Ep01Init = "1";

    var dx = root.querySelector("#sim-ep0601_dx");
    var dy = root.querySelector("#sim-ep0601_dy");
    var sz = root.querySelector("#sim-ep0601_sz");

    var vdx = root.querySelector("#sim-ep0601_vdx");
    var vdy = root.querySelector("#sim-ep0601_vdy");
    var vsz = root.querySelector("#sim-ep0601_vsz");

    var gRef  = root.querySelector("#sim-ep0601_ref");
    var gPred = root.querySelector("#sim-ep0601_pred");
    var gMix  = root.querySelector("#sim-ep0601_mix");

    var dbg = root.querySelector("#sim-ep0601_dbg");

    var N = 10;
    var ref = {x: 2, y: 2, w: 6, h: 6};

    function inside(x, y, r){
      return x >= r.x && x < r.x + r.w && y >= r.y && y < r.y + r.h;
    }

    function pixel(color){
      var d = document.createElement("div");
      d.className = "sim-ep0601_px";
      d.style.background = color;
      return d;
    }

    function classe(i){
      if (i >= 0.90) return "Excelente";
      if (i >= 0.75) return "Muito boa";
      if (i >= 0.50) return "Aceitável";
      return "Ruim";
    }

    function render(){
      vdx.textContent = dx.value;
      vdy.textContent = dy.value;
      vsz.textContent = sz.value;

      gRef.innerHTML  = "";
      gPred.innerHTML = "";
      gMix.innerHTML  = "";

      var pred = {
        x: ref.x + parseInt(dx.value, 10),
        y: ref.y + parseInt(dy.value, 10),
        w: parseInt(sz.value, 10),
        h: parseInt(sz.value, 10)
      };

      var inter = 0;
      var uniao = 0;

      for (var y = 0; y < N; y++){
        for (var x = 0; x < N; x++){
          var r = inside(x, y, ref);
          var p = inside(x, y, pred);

          gRef.appendChild(pixel(r ? "#7fdc92" : "#ffffff"));
          gPred.appendChild(pixel(p ? "#7fbfff" : "#ffffff"));

          if (r && p){
            gMix.appendChild(pixel("#9b59b6"));
            inter++;
          }
          else if (r){
            gMix.appendChild(pixel("#7fdc92"));
            uniao++;
          }
          else if (p){
            gMix.appendChild(pixel("#7fbfff"));
            uniao++;
          }
          else{
            gMix.appendChild(pixel("#ffffff"));
          }

          if (r && p) uniao++;
        }
      }

      var iou = inter / uniao;

      dbg.innerHTML =
        "<b>Interseção</b> = " + inter + " pixels &nbsp;&nbsp;&nbsp;" +
        "<b>União</b> = " + uniao + " pixels<br><br>" +
        "IoU = <b>" + inter + " / " + uniao + " = " + iou.toFixed(4) + "</b><br><br>" +
        "<span style='font-weight:700; color:#04342C;'>" + classe(iou) + "</span>";
    }

    dx.addEventListener('input', render);
    dy.addEventListener('input', render);
    sz.addEventListener('input', render);

    render();
  }

  function tryInitSim06Ep01(){
    var root = document.getElementById('sim-ep0601');
    if (root) initSim06Ep01(root); else setTimeout(tryInitSim06Ep01, 200);
  }
  tryInitSim06Ep01();
})();
</script>
""")

**Figura 6.1:** Simulatore EP06_01: IoU tra maschera di riferimento e maschera predetta


<figure id="fig-06-sim-ep0601">
  <img src="imagens/fig-06-sim-ep0601.png" alt=" Simulatore EP06_01: IoU tra maschera di riferimento e maschera predetta " style="max-width:80%" />
  <figcaption><strong>Figura 6.1:</strong>  Simulatore EP06_01: IoU tra maschera di riferimento e maschera predetta </figcaption>
</figure>

In [ ]:
%%writefile EP06_01.py
# Codice Python

In [ ]:
TestSuite("EP06_01.py").run()

### EP06_02 🟢 Filtro di Marcatori per Circolarità

Dopo la segmentazione di un’immagine, è comune che vengano identificati diversi componenti connessi. In applicazioni come la rettifica di documenti, solo alcuni di questi componenti corrispondono ai marcatori di riferimento utilizzati per l’allineamento dell’immagine. Un criterio spesso impiegato per selezionare tali marcatori è la **circolarità**, che misura quanto la forma di un componente sia vicina a un cerchio.

In questo esercizio, ogni componente è descritto dalla sua area $A$ e dal suo perimetro $P$. L’obiettivo è calcolarne la circolarità e decidere, in base a una soglia fornita, se il componente debba essere accettato o rifiutato come candidato marcatore.

#### 📋 Linee Guida di Implementazione

1. **Quantità:** Leggere l’intero $N$ (numero di candidati) e la soglia di circolarità $C_{\text{limiar}}$ (numero reale).
2. **Dati dei candidati:** Per ciascuno degli $N$ candidati, leggere l’area $A$ (intero) e il perimetro $P$ (numero reale).
3. **Circolarità:** Calcolare $C=\frac{4\pi A}{P^2}$, dove:

- $A$ è l’area del componente;
- $P$ è il perimetro del componente;
- $C$ è la circolarità.

4. **Caso degenere:** Se $P=0$, considerare $C=0$ e classificare direttamente il candidato come `REJEITADO`.
5. **Classificazione:** Se $C>C_{\text{limiar}}$, classificare il candidato come `ACEITO`; altrimenti, classificarlo come `REJEITADO`.
6. **Arrotondamento:** Visualizzare il valore di $C$ con quattro cifre decimali.
7. **Output:** Per ogni candidato, stampare il valore di $C$ seguito dalla classificazione. Alla fine, stampare il numero totale di candidati accettati.

#### 📌 Vincoli Computazionali

- Utilizzare la costante $\pi$ della libreria standard del linguaggio (ad esempio, `math.pi`), senza approssimazioni.
- Il confronto deve essere effettuato con il valore di $C$ a piena precisione, prima dell’arrotondamento per la visualizzazione.
- Il criterio di accettazione è stretto ($C>C_{\text{limiar}}$).
- Se $P=0$, la divisione non deve essere eseguita.

#### 🧠 Fondamenti Teorici

La circolarità è un descrittore geometrico definito da $C=\frac{4\pi A}{P^2}$, dove:

- $A$ è l’area del componente;
- $P$ è il perimetro del componente;
- $C$ è la circolarità.

Per un cerchio perfetto, $C=1$. Man mano che la forma diventa più allungata o irregolare, il perimetro cresce più rapidamente dell’area, riducendo il valore di $C$.

| Forma | Circolarità approssimata | Interpretazione |
|---|---:|---|
| Cerchio | $1{,}0000$ | Forma circolare. |
| Quadrato | $0{,}7854$ | Forma approssimativamente compatta. |
| Forma allungata o irregolare | $C\ll1$ | Bassa circolarità. |
| $P=0$ | $0$ (convenzione adottata) | Contorno degenere. |

La circolarità è invariante rispetto a traslazione, rotazione e scala, ed è ampiamente utilizzata per distinguere componenti approssimativamente circolari da altre forme.

#### 📦 Specifica di Input e Output (VPL)

**Input:**

- Riga 1: intero $N$.
- Riga 2: numero reale $C_{\text{limiar}}$.
- Successive $N$ righe: area $A$ (intero) e perimetro $P$ (reale), separati da spazio.

**Output:**

- Una riga per ogni candidato, nel formato `C ACEITO` o `C REJEITADO`, con $C$ presentato con quattro cifre decimali.
- Ultima riga: `Total aceitos: X`.

#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 3<br>0.6<br>78 31.4<br>100 40<br>50 60 | 0.9941 ACEITO<br>0.7854 ACEITO<br>0.1745 REJEITADO<br>Total aceitos: 2 | Candidato approssimativamente circolare, forma compatta e forma allungata. |
| 1<br>0.9<br>10 0 | 0.0000 REJEITADO<br>Total aceitos: 0 | Perimetro nullo: contorno degenere. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0602" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0602 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0602 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0602 button:hover { background: #e8dfcf; }
  #sim-ep0602 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0602_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0602_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP06_02: Filtro dei Marcatori per Circolarità</span>
  <span class="sim-ep0602_pill">C = 4&pi;A / P&sup2;</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0602_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Soglia di circolarità (C_soglia): <span id="sim-ep0602_vl" style="font-family:monospace; color:#26241d;">0.60</span>
      </label>
    </div>
    
    <input id="sim-ep0602_sl" type="range" min="0.05" max="0.99" step="0.01" value="0.60">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Regola la soglia e osserva quali candidati (dischi, quadrati e forme irregolari) sopravvivono al filtro.
    </div>
  </div>

  <!-- Cards de Candidatos -->
  <div id="sim-ep0602_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(100px, 1fr)); gap:10px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0602_debug" class="sim-ep0602_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep02(root){
    if (!root || root.dataset.sim06Ep02Init) return;
    root.dataset.sim06Ep02Init = "1";

    var candidatos = [
      {nome: "Disco", A: 78, P: 31.4},
      {nome: "Quadrado", A: 100, P: 40},
      {nome: "Retângulo", A: 60, P: 44},
      {nome: "Rasura", A: 50, P: 60},
      {nome: "Ponto", A: 10, P: 0}
    ];

    var slEl  = root.querySelector('#sim-ep0602_sl');
    var vlEl  = root.querySelector('#sim-ep0602_vl');
    var cards = root.querySelector('#sim-ep0602_cards');
    var dbg   = root.querySelector('#sim-ep0602_debug');

    function render(){
      var th = parseFloat(slEl.value);
      vlEl.textContent = th.toFixed(2);
      cards.innerHTML = '';
      var aceitos = 0;

      candidatos.forEach(function(c){
        var C = (c.P === 0) ? 0 : (4 * Math.PI * c.A) / (c.P * c.P);
        var ok = c.P !== 0 && C > th;
        if (ok) aceitos++;

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' +
          (ok ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;');
        
        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">' + c.nome + '</div>' +
          '<div style="font-family:monospace; margin-bottom:4px; font-size:10px; opacity:0.8;">A = ' + c.A + '<br>P = ' + c.P + '</div>' +
          '<div style="font-family:monospace; font-weight:700; margin-bottom:4px;">C = ' + C.toFixed(4) + '</div>' +
          '<div style="font-weight:700; font-size:10px; letter-spacing:0.04em;">' + (ok ? 'ACEITO' : 'REJEITADO') + '</div>';
        
        cards.appendChild(div);
      });

      dbg.textContent = 'C_soglia = ' + th.toFixed(2) + '  |  Candidatos aceitos: ' + aceitos + ' / ' + candidatos.length;
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep02(){
    var root = document.getElementById('sim-ep0602');
    if (root) initSim06Ep02(root); else setTimeout(tryInitSim06Ep02, 200);
  }
  tryInitSim06Ep02();
})();
</script>
""")

**Figura 6.2:** Simulatore EP06_02: Filtro di Marcatori per Circolarità


<figure id="fig-06-sim-ep0602">
  <img src="imagens/fig-06-sim-ep0602.png" alt=" Simulatore EP06_02: Filtro di Marcatori per Circolarità " style="max-width:80%" />
  <figcaption><strong>Figura 6.2:</strong>  Simulatore EP06_02: Filtro di Marcatori per Circolarità </figcaption>
</figure>

In [ ]:
%%writefile EP06_02.py
# Codice Python

In [ ]:
TestSuite("EP06_02.py").run()

### EP06_03 🟡 Classificazione delle Marcature nei Fogli di Risposta (OMR)

Dopo la rettifica del foglio e la segmentazione dei riquadri delle risposte, il MCTest stima, per ogni bolla, un **grado di riempimento**, rappresentato da un valore tra $0$ e $100$. Sulla base di questi valori, il sistema deve determinare automaticamente l'alternativa marcata, individuando anche domande in bianco e casi di marcature multiple.

In questo esercizio, implementerai questa fase di decisione del *pipeline* di OMR. La classificazione dipende da una soglia di riempimento: piccole variazioni di questo valore possono alterare il risultato della lettura automatica.

#### 📋 Linee Guida di Implementazione

1. **Parametri:** Leggere gli interi $Q$ (numero di domande) e $K$ (numero di alternative per domanda, con $2 \le K \le 26$) e la soglia di riempimento $\mathrm{Th}$ (numero reale tra $0$ e $100$).
2. **Gradi di riempimento:** Per ciascuna delle $Q$ domande, leggere i $K$ valori reali corrispondenti alle alternative `A`, `B`, `C`, ..., nell'ordine di ingresso.
3. **Conteggio delle marcature:** Per ogni domanda, contare quante alternative hanno un grado di riempimento **strettamente maggiore** di $\mathrm{Th}$.
4. **Classificazione:**
   - Se nessuna alternativa supera $\mathrm{Th}$, classificare la domanda come `BRANCO`.
   - Se esattamente un'alternativa supera $\mathrm{Th}$, stampare la lettera corrispondente (`A`, `B`, `C`, ...).
   - Se due o più alternative superano $\mathrm{Th}$, classificare la domanda come `DUPLA_MARCACAO`.
5. **Output per domanda:** Stampare, nell'ordine di lettura, la classificazione di ciascuna domanda.
6. **Totali:** Alla fine, stampare il numero di domande `OK` (marcatura singola), `BRANCO` e `DUPLA_MARCACAO`.

#### 📌 Vincoli Computazionali

* **Confronto stretto:** solo i valori maggiori di $\mathrm{Th}$ sono considerati marcature valide; i valori esattamente uguali alla soglia non devono essere conteggiati.
* **Lettere delle alternative:** l'indice $0$ corrisponde all'alternativa `A`, l'indice $1$ all'alternativa `B` e così via.
* **Marcature multiple:** ogni volta che due o più alternative superano la soglia, la classificazione deve essere `DUPLA_MARCACAO`, indipendentemente dai rispettivi gradi di riempimento.

#### 🧠 Fondamento Teorico

| Situazione | Classificazione | Interpretazione |
|---|---|---|
| Esattamente un'alternativa sopra la soglia | Lettera dell'alternativa | Risposta valida |
| Nessuna alternativa sopra la soglia | `BRANCO` | Domanda senza risposta |
| Due o più alternative sopra la soglia | `DUPLA_MARCACAO` | Risposta ambigua |

La soglia di riempimento controlla la sensibilità dell'algoritmo. Valori molto bassi tendono ad aumentare il numero di `DUPLA_MARCACAO`, mentre valori molto alti possono aumentare la quantità di domande classificate come `BRANCO`.

#### 📦 Specifica di Ingresso e Uscita (VPL)

**Ingresso:**

* Riga 1: Intero $Q$.
* Riga 2: Intero $K$.
* Riga 3: Numero reale $\mathrm{Th}$.
* Prossime $Q$ righe: $K$ numeri reali, corrispondenti ai gradi di riempimento delle alternative.
* Riga 1: Interi $Q$ e $K$.

**Uscita:**

* $Q$ righe, ciascuna contenente la classificazione della rispettiva domanda.
* Riga finale: `OK: x  BRANCO: y  DUPLA_MARCACAO: z`.

#### 📌 Esempi

| Ingresso | Uscita | Osservazione |
|---|---|---|
| 3<br>4<br>50<br>10 85 5 12<br>20 15 18 22<br>90 88 10 5 | B<br>BRANCO<br>DUPLA_MARCACAO<br>OK: 1  BRANCO: 1  DUPLA_MARCACAO: 1 | Nella prima domanda solo `B` supera la soglia; nella seconda nessuna alternativa la supera; nella terza, `A` e `B` superano la soglia. |
| 1<br>2<br>50.0<br>50 50 | BRANCO<br>OK: 0  BRANCO: 1  DUPLA_MARCACAO: 0 | I valori uguali alla soglia non sono considerati marcature valide. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0603" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0603 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0603 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0603 button:hover { background: #e8dfcf; }
  #sim-ep0603 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0603_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0603_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP06_03: Classificazione delle Marcature OMR</span>
  <span class="sim-ep0603_pill">4 Alternative</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0603_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Soglia di Compilazione (Th): <span id="sim-ep0603_vth" style="font-family:monospace; color:#26241d;">50</span>%
      </label>
    </div>
    
    <input id="sim-ep0603_th" type="range" min="0" max="100" step="1" value="50">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Regola il grado di compilazione di ciascuna bolla (A&ndash;D) e la soglia per osservare la classificazione risultante.
    </div>
  </div>

  <!-- Sliders das Bolhas (A-D) -->
  <div class="sim-ep0603_panel" style="margin-bottom:14px;">
    <div id="sim-ep0603_bubbles" style="display:grid; grid-template-columns:repeat(4, 1fr); gap:12px;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0603_debug" class="sim-ep0603_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep03(root){
    if (!root || root.dataset.sim06Ep03Init) return;
    root.dataset.sim06Ep03Init = "1";

    var letras = ['A', 'B', 'C', 'D'];
    var valores = [10, 85, 5, 12];
    var thEl  = root.querySelector('#sim-ep0603_th');
    var vthEl = root.querySelector('#sim-ep0603_vth');
    var box   = root.querySelector('#sim-ep0603_bubbles');
    var dbg   = root.querySelector('#sim-ep0603_debug');

    box.innerHTML = '';
    var sliders = [];

    letras.forEach(function(L, i){
      var col = document.createElement('div');
      col.style.cssText = 'text-align:center; background:#fafaf7; border:1px solid #e9e3d3; padding:10px; border-radius:8px;';
      col.innerHTML = '<div style="font-weight:700; font-size:12px; color:#5e5a4a; margin-bottom:6px;">' + L + '</div>' +
        '<input type="range" min="0" max="100" step="1" value="' + valores[i] + '" id="sim-ep0603_b' + i + '">' +
        '<div id="sim-ep0603_v' + i + '" style="font-family:monospace; font-weight:700; font-size:11px; color:#26241d; margin-top:6px;">' + valores[i] + '%</div>';
      box.appendChild(col);
      sliders.push(col.querySelector('#sim-ep0603_b' + i));
    });

    function render(){
      var th = parseFloat(thEl.value);
      vthEl.textContent = th.toFixed(0);
      var marcadas = [];

      sliders.forEach(function(s, i){
        var v = parseFloat(s.value);
        root.querySelector('#sim-ep0603_v' + i).textContent = v.toFixed(0) + '%';
        if (v > th) marcadas.push(letras[i]);
      });

      var resultado;
      if (marcadas.length === 0) {
        resultado = 'BRANCO';
        dbg.style.borderColor = '#e4dcc8';
        dbg.style.background  = '#fafaf7';
        dbg.style.color       = '#8a8371';
      } else if (marcadas.length === 1) {
        resultado = 'RESPOSTA: ' + marcadas[0];
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        resultado = 'DUPLA_MARCACAO (' + marcadas.join(', ') + ')';
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = 'Classificazione della domanda: ' + resultado;
    }

    sliders.forEach(function(s){ s.addEventListener('input', render); });
    thEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep03(){
    var root = document.getElementById('sim-ep0603');
    if (root) initSim06Ep03(root); else setTimeout(tryInitSim06Ep03, 200);
  }
  tryInitSim06Ep03();
})();
</script>
""")

**Figura 6.3:** Simulatore EP06_03: Classificazione delle Marcature OMR


<figure id="fig-06-sim-ep0603">
  <img src="imagens/fig-06-sim-ep0603.png" alt=" Simulatore EP06_03: Classificazione delle Marcature OMR " style="max-width:80%" />
  <figcaption><strong>Figura 6.3:</strong>  Simulatore EP06_03: Classificazione delle Marcature OMR </figcaption>
</figure>

In [ ]:
%%writefile EP06_03.py
# Codice Python

In [ ]:
TestSuite("EP06_03.py").run()

### EP06_04 🟡 Stimatore dell'Inclinazione tramite Mediana Angolare (*Deskew*)

Dopo il rilevamento dei bordi e l'applicazione della Trasformata di Hough, si ottiene un insieme di rette candidate all'orientamento predominante del documento. Ogni retta fornisce una stima dell'angolo di inclinazione, calcolata come

$$
\text{angolo} = \operatorname{rad2deg}(\theta) - 90.
$$

Tuttavia, non tutte le rette corrispondono alle righe del documento: alcune derivano da rumori, ombre o altri elementi dell'immagine. In questo esercizio, implementerai la fase di stima robusta dell'angolo di inclinazione, filtrando i valori plausibili e calcolandone la mediana.

#### 📋 Linee Guida di Implementazione

1. **Quantità:** Leggere l'intero $M$, corrispondente al numero di angoli stimati.
2. **Angoli:** Leggere gli $M$ valori reali, in gradi.
3. **Filtraggio:** Mantenere solo gli angoli che soddisfano **strettamente** $-45 < \text{angolo} < 45$.
4. **Assenza di candidati:** Se nessun angolo rimane dopo il filtraggio, stampare esattamente `SEM_CORRECAO`.
5. **Mediana:** Nel caso in cui esistano angoli validi:
   - se la quantità è dispari, la mediana è l'elemento centrale della sequenza ordinata;
   - se è pari, la mediana è la media aritmetica dei due elementi centrali.
6. **Uscita:** Stampare la mediana arrotondata a due cifre decimali (arrotondamento standard, *round half away from zero*, con `np.floor(img + 0.5)`).

#### 📌 Vincoli Computazionali

* **Intervallo aperto:** angoli uguali a $-45$ o $45$ non devono essere considerati.
* **Precisione:** calcolare la mediana utilizzando i valori originali; l'arrotondamento deve essere effettuato solo in uscita.
* **Caso vuoto:** se non ci sono angoli validi, non deve essere calcolata alcuna mediana.

#### 🧠 Fondamenti Teorici

| Situazione | Risultato |
|---|---|
| Maggior parte degli angoli concentrata attorno all'inclinazione reale | La mediana approssima l'orientamento del documento. |
| Pochi angoli discordanti (*outliers*) | La mediana subisce poca influenza da questi valori. |
| Angoli al di fuori dell'intervallo $(-45^\circ,45^\circ)$ | Vengono scartati prima del calcolo. |
| Nessun angolo valido | Non viene applicata alcuna correzione (`SEM_CORRECAO`). |

La mediana viene utilizzata perché è più robusta della media in presenza di pochi valori discordanti, producendo una stima più stabile dell'inclinazione predominante del documento.

#### 📦 Specifica di Ingresso e Uscita (VPL)

**Ingresso:**

* Riga 1: Intero $M$.
* Riga 2: $M$ numeri reali, corrispondenti agli angoli in gradi.

**Uscita:**

* Una singola riga contenente l'angolo stimato, con due cifre decimali, oppure la parola `SEM_CORRECAO` se nessun angolo è valido.

#### 📌 Esempi

| Ingresso | Uscita | Osservazione |
|---|---|---|
| 5<br>-50 -10.5 2.3 2.3 47 | 2.30 | Solo gli angoli nell'intervallo $(-45,45)$ sono considerati; la mediana è $2{,}3$. |
| 4<br>-46 50 45 -45 | SEM_CORRECAO | Nessun angolo appartiene all'intervallo aperto $(-45,45)$. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0604" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0604 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0604 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0604 button:hover { background: #e8dfcf; }
  #sim-ep0604 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0604_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0604_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP06_04: Stimatore di Inclinazione per Mediana Angolare (Deskew)</span>
  <span class="sim-ep0604_pill">mediana(-45&deg; &lt; &theta; &lt; 45&deg;)</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0604_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Angolo del rumore extra (&theta;_rumore): <span id="sim-ep0604_vl" style="font-family:monospace; color:#26241d;">47</span>&deg;
      </label>
    </div>
    
    <input id="sim-ep0604_sl" type="range" min="-80" max="80" step="1" value="47">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Trascina l'angolo del rumore extra dentro o fuori dall'intervallo [-45&deg;, +45&deg;] e osserva come la mediana rimane stabile.
    </div>
  </div>

  <!-- Exibição dos Ângulos Amostrados -->
  <div class="sim-ep0604_panel" style="margin-bottom:14px;">
    <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; text-align:center; letter-spacing:0.04em;">
      Campioni di Angoli (Verde = Dentro l'Intervallo, Rosso = Rumore Scartato)
    </div>
    <div id="sim-ep0604_pts" style="display:flex; gap:8px; flex-wrap:wrap; justify-content:center;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0604_debug" class="sim-ep0604_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep04(root){
    if (!root || root.dataset.sim06Ep04Init) return;
    root.dataset.sim06Ep04Init = "1";

    var base = [-10.5, 2.3, 2.3];
    var slEl  = root.querySelector('#sim-ep0604_sl');
    var vlEl  = root.querySelector('#sim-ep0604_vl');
    var ptsEl = root.querySelector('#sim-ep0604_pts');
    var dbg   = root.querySelector('#sim-ep0604_debug');

    function median(arr){
      var a = arr.slice().sort(function(x, y){ return x - y; });
      var n = a.length;
      if (n === 0) return null;
      var mid = Math.floor(n / 2);
      return (n % 2 === 1) ? a[mid] : (a[mid - 1] + a[mid]) / 2;
    }

    function render(){
      var extra = parseFloat(slEl.value);
      vlEl.textContent = extra;
      var todos = base.concat([extra, -50]);
      var validos = todos.filter(function(a){ return a > -45 && a < 45; });
      
      ptsEl.innerHTML = '';
      todos.forEach(function(a){
        var ok = a > -45 && a < 45;
        var div = document.createElement('div');
        div.style.cssText = 'padding:8px 12px; border-radius:8px; font-family:monospace; font-size:12px; font-weight:700; transition:all 0.15s ease;' +
          (ok ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;');
        div.textContent = a + '°';
        ptsEl.appendChild(div);
      });

      var med = median(validos);

      if (med === null) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
        dbg.textContent = 'Válidos: []  |  Mediana estimada: SEM_CORRECAO';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
        dbg.textContent = 'Validi: [' + validos.join(', ') + ']  |  Mediana estimada: ' + med.toFixed(2) + '°';
      }
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep04(){
    var root = document.getElementById('sim-ep0604');
    if (root) initSim06Ep04(root); else setTimeout(tryInitSim06Ep04, 200);
  }
  tryInitSim06Ep04();
})();
</script>
""")

**Figura 6.4:** Simulatore EP06_04: Stimatore della Pendenza per Mediana Angolare


<figure id="fig-06-sim-ep0604">
  <img src="imagens/fig-06-sim-ep0604.png" alt=" Simulatore EP06_04: Stimatore della Pendenza per Mediana Angolare " style="max-width:80%" />
  <figcaption><strong>Figura 6.4:</strong>  Simulatore EP06_04: Stimatore della Pendenza per Mediana Angolare </figcaption>
</figure>

In [ ]:
%%writefile EP06_04.py
# Codice Python

In [ ]:
TestSuite("EP06_04.py").run()

### EP06_05 🟠 Normalizzazione del fondo per divisione (Correzione dell'illuminazione)

Un modulo è stato fotografato in condizioni di illuminazione non uniforme, facendo sì che un lato del foglio appaia più chiaro dell'altro. In queste condizioni, la sogliatura globale di Otsu può produrre risultati insoddisfacenti, poiché una singola soglia non separa adeguatamente testo e sfondo in tutta l'immagine. La soluzione presentata nel capitolo consiste nel **normalizzare lo sfondo**, dividendo l'immagine originale per una versione fortemente smussata di se stessa, che rappresenta l'illuminazione a bassa frequenza.

In questo esercizio, l'immagine originale e lo sfondo smussato (equivalente al risultato di un `cv2.GaussianBlur` con $\sigma$ elevato) sono già forniti. Il tuo compito è implementare la fase di normalizzazione che produce l'immagine corretta.

#### 📋 Linee guida di implementazione

1. **Dimensioni:** Leggere gli interi $L$ (righe) e $C$ (colonne).
2. **Immagine originale:** Leggere i valori interi $L \times C$ della matrice `img` (intensità tra 0 e 255).
3. **Sfondo stimato:** Leggere i valori interi $L \times C$ della matrice `bg` (intensità tra 0 e 255, sempre strettamente maggiori di zero).
4. **Normalizzazione:** Per ogni posizione $(i,j)$, calcolare
$$
\text{valore}(i,j)=
\frac{\text{img}(i,j)}{\text{bg}(i,j)}\times255.
$$
5. **Arrotondamento:** Arrotondare il risultato all'intero più vicino (*round half away from zero*, con `np.floor(img + 0.5)`).
6. **Saturazione:** Limitare il valore ottenuto all'intervallo $[0,255]$.
7. **Uscita:** Stampare la matrice `img_norm` risultante.

#### 📌 Vincoli computazionali

* **Divisione per zero:** l'input garantisce $\text{bg}(i,j)>0$ in tutte le posizioni.
* **Ordine delle operazioni:** prima arrotondare, poi applicare la saturazione.
* **Elaborazione indipendente:** ogni pixel deve essere normalizzato singolarmente, senza utilizzare informazioni dai pixel vicini.

#### 🧠 Fondamenti teorici

| Situazione | Effetto della normalizzazione |
|---|---|
| $\text{img}(i,j)=\text{bg}(i,j)$ | Risultato pari a $255$, corrispondente allo sfondo normalizzato. |
| $\text{img}(i,j)<\text{bg}(i,j)$ | Risultato inferiore a $255$, preservando le regioni più scure, come il testo. |
| $\text{img}(i,j)>\text{bg}(i,j)$ | Risultato superiore a $255$, successivamente saturato. |
| Sfondo con illuminazione non uniforme | La divisione riduce le variazioni lente dell'illuminazione, rendendo l'immagine più omogenea. |

La divisione per lo sfondo stimato riduce gli effetti dell'illuminazione non uniforme e preserva il contrasto tra primo piano e sfondo, facilitando le fasi successive di segmentazione.

#### 📦 Specifica di input e output (VPL)

**Input:**

* Riga 1: Intero $L$.
* Riga 2: Intero $C$.
* Prossime $L$ righe: elementi della matrice `img`.
* Prossime $L$ righe: elementi della matrice `bg`.

**Output:**

* Matrice `img_norm`, con $L$ righe e $C$ colonne, contenente valori interi separati da spazi.

#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 2<br>2<br>60 120<br>180 40<br>100 100<br>200 80 | 153 255<br>230 128 | I valori superiori a $255$ devono essere saturati; $180/200\times255=229{,}5$ risulta in $230$ dopo l'arrotondamento. |
| 1<br>3<br>30 60 90<br>60 60 60 | 128 255 255 | Solo il primo valore rimane al di sotto di $255$ dopo la normalizzazione. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0605" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0605 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0605 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0605 button:hover { background: #e8dfcf; }
  #sim-ep0605 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0605_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0605_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim05_ep05_cell { width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 9px; font-weight: 700; font-family: monospace; border: 1px solid #e4dcc8; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP06_05: Normalizzazione dello Sfondo per Divisione</span>
  <span class="sim-ep0605_pill">(img / bg) &times; 255</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0605_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Intensità dello Sfondo a Sinistra (bg_sx): <span id="sim-ep0605_vl" style="font-family:monospace; color:#26241d;">100</span>
      </label>
    </div>
    
    <input id="sim-ep0605_sl" type="range" min="40" max="220" step="5" value="100">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Regola il gradiente di sfondo (sinistra &rarr; destra) e osserva come la divisione annulla la variazione di illuminazione.
    </div>
  </div>

  <!-- Exibição das Matrizes 1x4 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(160px, 1fr)); gap:12px; margin-bottom:14px;">
    
    <div class="sim-ep0605_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        img (Originale)
      </div>
      <div id="sim-ep0605_g_img" style="display:grid; grid-template-columns:repeat(4, 40px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0605_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        bg (Sfondo Attenuato)
      </div>
      <div id="sim-ep0605_g_bg" style="display:grid; grid-template-columns:repeat(4, 40px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0605_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        img_norm (Uscita)
      </div>
      <div id="sim-ep0605_g_out" style="display:grid; grid-template-columns:repeat(4, 40px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0605_debug" class="sim-ep0605_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep05(root){
    if (!root || root.dataset.sim06Ep05Init) return;
    root.dataset.sim06Ep05Init = "1";

    var linha_img = [90, 90, 90, 90];
    var slEl = root.querySelector('#sim-ep0605_sl');
    var vlEl = root.querySelector('#sim-ep0605_vl');
    var gImg = root.querySelector('#sim-ep0605_g_img');
    var gBg  = root.querySelector('#sim-ep0605_g_bg');
    var gOut = root.querySelector('#sim-ep0605_g_out');
    var dbg  = root.querySelector('#sim-ep0605_debug');

    function roundHalfAway(x){
      return x >= 0 ? Math.floor(x + 0.5) : Math.ceil(x - 0.5);
    }

    function cellStyle(v){
      var g = Math.max(0, Math.min(255, v));
      return 'background:rgb(' + g + ',' + g + ',' + g + '); color:' + (g > 140 ? '#000000' : '#ffffff') + ';';
    }

    function render(){
      var bgEsq = parseInt(slEl.value, 10);
      vlEl.textContent = bgEsq;

      // Gradiente linear de bgEsq até 200 na direita, 4 colunas
      var bg = [];
      for (var j = 0; j < 4; j++){
        bg.push(Math.round(bgEsq + (200 - bgEsq) * j / 3));
      }

      gImg.innerHTML = '';
      gBg.innerHTML  = '';
      gOut.innerHTML = '';
      
      var out = [];
      for (var j = 0; j < 4; j++){
        var v = (linha_img[j] / bg[j]) * 255;
        var r = roundHalfAway(v);
        var sat = Math.max(0, Math.min(255, r));
        out.push(sat);

        var ci = document.createElement('div');
        ci.className = 'sim05_ep05_cell';
        ci.style.cssText = cellStyle(linha_img[j]);
        ci.textContent = linha_img[j];
        gImg.appendChild(ci);

        var cb = document.createElement('div');
        cb.className = 'sim05_ep05_cell';
        cb.style.cssText = cellStyle(bg[j]);
        cb.textContent = bg[j];
        gBg.appendChild(cb);

        var co = document.createElement('div');
        co.className = 'sim05_ep05_cell';
        co.style.cssText = cellStyle(sat);
        co.textContent = sat;
        gOut.appendChild(co);
      }

      dbg.textContent = 'bg = [' + bg.join(', ') + ']  |  img_norm = [' + out.join(', ') + ']';
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep05(){
    var root = document.getElementById('sim-ep0605');
    if (root) initSim06Ep05(root); else setTimeout(tryInitSim06Ep05, 200);
  }
  tryInitSim06Ep05();
})();
</script>
""")

**Figura 6.5:** Simulatore EP06_05: Normalizzazione dello Sfondo per Divisione


<figure id="fig-06-sim-ep0605">
  <img src="imagens/fig-06-sim-ep0605.png" alt=" Simulatore EP06_05: Normalizzazione dello Sfondo per Divisione " style="max-width:80%" />
  <figcaption><strong>Figura 6.5:</strong>  Simulatore EP06_05: Normalizzazione dello Sfondo per Divisione </figcaption>
</figure>

In [ ]:
%%writefile EP06_05.py
# Codice Python

In [ ]:
TestSuite("EP06_05.py").run()

### EP06_06 🔴 Mappa di Varianza Locale per il Rilevamento di Trama

Una fabbrica tessile deve ispezionare rotoli di tessuto in tempo reale, senza disporre di un'immagine di riferimento — ogni rotolo presenta piccole variazioni naturali. In questa situazione, la strategia presentata nel capitolo consiste nell'analizzare l'**omogeneità locale della trama**: le regioni uniformi presentano una bassa varianza di intensità in piccoli intorni, mentre graffi, macchie e difetti di fabbricazione producono aumenti locali di tale varianza.

In questo esercizio, implementerai il nucleo di questo metodo, calcolando la varianza locale in una finestra scorrevole e generando una maschera binaria che identifica le regioni la cui varianza supera una soglia.

#### 📋 Linee Guida di Implementazione

1. **Dimensioni e parametri:** Leggere gli interi $L$, $C$, $k$ (dimensione della finestra, sempre dispari) e $T$ (soglia di varianza).
2. **Immagine:** Leggere gli $L \times C$ valori interi della matrice di trama (intensità tra 0 e 255).
3. **Gestione dei bordi:** Quando la finestra supera i limiti dell'immagine, utilizzare la **replicazione del bordo**, cioè ripetere il valore del pixel valido più vicino.
4. **Media locale:** Per ogni posizione $(i,j)$, calcolare
$$
\mu(i,j)=
\frac{1}{k^2}
\sum_{(p,q)\in\text{finestra}}
\text{trama}(p,q).
$$
5. **Varianza locale:** Calcolare la varianza della popolazione della finestra,
$$
\sigma^2(i,j)=
\frac{1}{k^2}
\sum_{(p,q)\in\text{finestra}}
\left(\text{trama}(p,q)-\mu(i,j)\right)^2,
$$
o, in modo equivalente,
$$
\sigma^2(i,j)=\overline{x^2}-\mu(i,j)^2,
$$
dove $\overline{x^2}$ rappresenta la media dei quadrati delle intensità.

6. **Arrotondamento:** Arrotondare la varianza all'intero più vicino (*round half away from zero*, con `np.floor(res_norm + 0.5)`).

7. **Soglia:** Definire $\text{maschera}(i,j)=1$ se la varianza arrotondata è **strettamente maggiore** di $T$; altrimenti, definire $\text{maschera}(i,j)=0$.

8. **Output:** Stampare la maschera binaria risultante.

#### 📌 Vincoli Computazionali

* **Replicazione del bordo:** utilizzare il valore del pixel valido più vicino ogni volta che la finestra supera i limiti dell'immagine.
* **Varianza della popolazione:** utilizzare il denominatore $k^2$, mai $k^2-1$.
* **Confronto stretto:** la maschera deve essere calcolata utilizzando la condizione $\sigma^2_{\text{arrotondata}}>T$.
* **Finestra dispari:** il valore di $k$ è sempre dispari, garantendo un pixel centrale.

#### 🧠 Fondamento Teorico

| Situazione | Varianza locale | Interpretazione |
|---|---|---|
| Regione uniforme | Bassa | Intensità simili nell'intorno. |
| Regione con difetto | Alta | La presenza di intensità distinte aumenta la dispersione dei valori. |
| Finestra piccola | Maggiore sensibilità a dettagli e rumore | Rileva alterazioni localizzate. |
| Finestra grande | Risposta più uniforme | Evidenzia difetti più grandi, ma riduce la precisione della loro localizzazione. |

La varianza locale misura la dispersione delle intensità in un intorno. Le regioni omogenee presentano una bassa varianza, mentre le alterazioni nella trama aumentano questa misura, consentendo di identificare possibili difetti tramite una semplice sogliatura.

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Intero $L$.
* Riga 2: Intero $C$.
* Riga 3: Intero $k$ (dispari).
* Riga 4: Intero $T$.
* Prossime $L$ righe: elementi interi della matrice di trama.

**Output:**

* Maschera binaria (valori 0 o 1), con $L$ righe e $C$ colonne.

#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 3<br>3<br>3<br>50<br>10 10 10<br>10 10 10<br>10 90 10 | 0 0 0<br>1 1 1<br>1 1 1 | Il difetto aumenta la varianza in tutte le finestre che lo contengono. |
| 2<br>2<br>3<br>5<br>100 100<br>100 100 | 0 0<br>0 0 | La trama è uniforme; la varianza è nulla in tutta l'immagine. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0606" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0606 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0606 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0606 button:hover { background: #e8dfcf; }
  #sim-ep0606 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0606_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0606_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0606_cell { width: 44px; height: 44px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; border: 1px solid #e4dcc8; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP06_06: Varianza Locale (Rilevamento della Trama)</span>
  <span class="sim-ep0606_pill">&sigma;&sup2; = m&eacute;dia(x&sup2;) &minus; m&eacute;dia(x)&sup2;</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0606_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Intensità del Difetto (Posizione Centrale): <span id="sim-ep0606_vl_def" style="font-family:monospace; color:#26241d;">90</span>
      </label>
    </div>
    <input id="sim-ep0606_sl_def" type="range" min="10" max="255" step="5" value="90">

    <div style="display:flex; justify-content:space-between; align-items:center; margin:10px 0 4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Soglia (T): <span id="sim-ep0606_vl_t" style="font-family:monospace; color:#26241d;">50</span>
      </label>
    </div>
    <input id="sim-ep0606_sl_t" type="range" min="0" max="2000" step="10" value="50">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Regola il valore del difetto e la soglia T; osserva come la finestra 3&times;3 diffonde il rilevamento nella vicinanza.
    </div>
  </div>

  <!-- Exibição das Grades 3x3 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0606_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Trama (3&times;3)
      </div>
      <div id="sim-ep0606_g_tex" style="display:grid; grid-template-columns:repeat(3, 44px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0606_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Maschera di Difetto
      </div>
      <div id="sim-ep0606_g_mask" style="display:grid; grid-template-columns:repeat(3, 44px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0606_debug" class="sim-ep0606_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep06(root){
    if (!root || root.dataset.sim06Ep06Init) return;
    root.dataset.sim06Ep06Init = "1";

    var slDef = root.querySelector('#sim-ep0606_sl_def');
    var vlDef = root.querySelector('#sim-ep0606_vl_def');
    var slT   = root.querySelector('#sim-ep0606_sl_t');
    var vlT   = root.querySelector('#sim-ep0606_vl_t');
    var gTex  = root.querySelector('#sim-ep0606_g_tex');
    var gMask = root.querySelector('#sim-ep0606_g_mask');
    var dbg   = root.querySelector('#sim-ep0606_debug');

    function roundHalfAway(x){
      return x >= 0 ? Math.floor(x + 0.5) : Math.ceil(x - 0.5);
    }

    function clampIdx(v, n){
      return Math.max(0, Math.min(n - 1, v));
    }

    function render(){
      var defeito = parseInt(slDef.value, 10);
      var T       = parseInt(slT.value, 10);
      vlDef.textContent = defeito;
      vlT.textContent   = T;

      var N = 3;
      var tex = [[10, 10, 10], [10, defeito, 10], [10, 10, 10]];

      gTex.innerHTML  = '';
      gMask.innerHTML = '';
      
      var mask = [];
      for (var i = 0; i < N; i++){
        var row = [];
        for (var j = 0; j < N; j++){
          var vals = [];
          for (var di = -1; di <= 1; di++){
            for (var dj = -1; dj <= 1; dj++){
              var pi = clampIdx(i + di, N);
              var pj = clampIdx(j + dj, N);
              vals.push(tex[pi][pj]);
            }
          }
          var mean = vals.reduce(function(a, b){ return a + b; }, 0) / vals.length;
          var meanSq = vals.reduce(function(a, b){ return a + b * b; }, 0) / vals.length;
          var varr = meanSq - mean * mean;
          var varRound = roundHalfAway(varr);
          row.push(varRound > T ? 1 : 0);
        }
        mask.push(row);
      }

      var total = 0;
      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var g = tex[i][j];
          var ct = document.createElement('div');
          ct.className = 'sim-ep0606_cell';
          ct.style.cssText = 'background:rgb(' + g + ',' + g + ',' + g + '); color:' + (g > 140 ? '#000000' : '#ffffff') + ';';
          ct.textContent = g;
          gTex.appendChild(ct);

          var m = mask[i][j];
          if (m) total++;

          var cm = document.createElement('div');
          cm.className = 'sim-ep0606_cell';
          if (m) {
            cm.style.cssText = 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
          } else {
            cm.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
          }
          cm.textContent = m;
          gMask.appendChild(cm);
        }
      }

      if (total > 0) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      }

      dbg.textContent = 'difetto = ' + defeito + '  |  T = ' + T + '  |  Pixels marcados: ' + total + ' / 9';
    }

    slDef.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep06(){
    var root = document.getElementById('sim-ep0606');
    if (root) initSim06Ep06(root); else setTimeout(tryInitSim06Ep06, 200);
  }
  tryInitSim06Ep06();
})();
</script>
""")

**Figura 6.6:** Simulatore EP06_06: Mappa della Varianza Locale per il Rilevamento della Trama


<figure id="fig-06-sim-ep0606">
  <img src="imagens/fig-06-sim-ep0606.png" alt=" Simulatore EP06_06: Mappa della Varianza Locale per il Rilevamento della Trama " style="max-width:80%" />
  <figcaption><strong>Figura 6.6:</strong>  Simulatore EP06_06: Mappa della Varianza Locale per il Rilevamento della Trama </figcaption>
</figure>

In [ ]:
%%writefile EP06_06.py
# Codice Python

In [ ]:
TestSuite("EP06_06.py").run()

### EP06_07 🟣 *Pipeline* di Ispezione Industriale: Registrazione per Traslazione e Sottrazione

In una linea di produzione, una telecamera fissa fotografa ogni pezzo che passa sul nastro trasportatore, confrontandolo con un'immagine di riferimento priva di difetti. Il problema: piccole vibrazioni del nastro spostano il pezzo rispetto alla posizione di riferimento a ogni acquisizione. Se la sottrazione di immagini viene applicata direttamente, senza correzione, lo spostamento di per sé genera già differenze enormi — **falsi positivi** che mascherano i difetti reali.

Questo è l'esercizio più completo del capitolo: devi **prima registrare** (allineare geometricamente) l'immagine acquisita usando uno spostamento noto $(dx, dy)$, fornito da un sensore di posizione del nastro, e **solo successivamente applicare la sottrazione** con sogliatura, esattamente come descritto nella sezione sull'ispezione industriale.

#### 📋 Linee Guida di Implementazione

1. **Dimensioni e parametri:** Leggere $L$, $C$ (dimensioni delle immagini), lo spostamento intero noto $dx, dy$ (che può essere negativo) e la soglia di rilevamento $T$ (intero).
2. **Immagini:** Leggere la matrice di riferimento (`ref`, $L\times C$, senza difetti) e la matrice acquisita (`cap`, $L\times C$, possibilmente spostata e con difetti).
3. **Registrazione per traslazione:** Costruire l'immagine allineata `alin` applicando lo spostamento $(dx,dy)$ ricevuto:
$$
\text{alin}(i,j) = \begin{cases} \text{cap}(i+dy,\; j+dx), & \text{se } (i+dy,\ j+dx) \in [0,L)\times[0,C) \\ 0, & \text{altrimenti} \end{cases}
$$
4. **Riempimento dei bordi:** Le posizioni che "escono" dall'immagine acquisita dopo lo spostamento ricevono il valore **0** (*zero-padding* — al di fuori del campo visivo della telecamera; **nota che questo esercizio usa zero, diversamente dalla replicazione dei bordi dell'EP06_06**).
5. **Differenza assoluta:** Calcolare, pixel per pixel,
$$
\text{diff}(i,j) = |\text{ref}(i,j) - \text{alin}(i,j)|
$$
6. **Sogliatura:** Definire $\text{maschera}(i,j) = 1$ se $\text{diff}(i,j) > T$; altrimenti, $\text{maschera}(i,j) = 0$.
7. **Output:** In questo ordine — (a) la matrice `alin` ($L\times C$); (b) la maschera dei difetti ($L\times C$); (c) un'ultima riga con il totale dei pixel classificati come difettosi.

#### 📌 Vincoli Computazionali

* ***Zero-padding*, non replicazione:** le posizioni al di fuori dei limiti dell'immagine acquisita, dopo lo spostamento, valgono esattamente 0 — questo è il punto che differenzia maggiormente questo esercizio dall'EP06_06.
* **Confronto stretto:** $\text{diff}(i,j) > T$.
* **Segno di $(dx,dy)$:** lo spostamento può essere positivo o negativo; la formula del passo 3 deve essere applicata letteralmente, senza invertire i segni.
* **Tutti i valori sono interi:** non c'è arrotondamento in questa fase.

#### 🧠 Fondamentazione Teorica

| Fase omessa | Conseguenza |
|---|---|
| Saltare la registrazione geometrica | L'intero bordo dell'immagine (introdotto dallo spostamento) viene marcato come "difetto" — falso positivo sistematico |
| Registrazione con $(dx,dy)$ errato | Il pezzo e il riferimento rimangono disallineati; la sottrazione rileva contorni spostati, non difetti reali |
| Soglia $T$ troppo bassa | Il rumore di acquisizione (variazioni di 1–2 livelli di grigio) viene confuso con un difetto |
| Soglia $T$ troppo alta | I difetti sottili non vengono rilevati |

La registrazione geometrica e la sottrazione sono fasi complementari: la prima garantisce che entrambe le immagini rappresentino esattamente la stessa scena nello stesso riferimento spaziale; la seconda isola ciò che è realmente cambiato tra di esse — idealmente, solo i difetti.

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Intero $L$.
* Riga 2: Intero $C$.
* Riga 3: Due interi $dx$ e $dy$, separati da uno spazio.
* Riga 4: Intero $T$.
* Prossime $L$ righe: elementi interi della matrice `ref`.
* Prossime $L$ righe: elementi interi della matrice `cap`.

**Output:**

* $L$ righe con la matrice `alin`.
* $L$ righe con la maschera dei difetti (0/1).
* Ultima riga: `Totale di pixel difettosi: X`.

#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 3<br>3<br>1 0<br>30<br>50 50 50<br>50 50 50<br>50 50 50<br>0 50 50<br>0 50 90<br>0 50 50 | 50 50 0<br>50 90 0<br>50 50 0<br>0 0 1<br>0 1 1<br>0 0 1<br>Totale di pixel difettosi: 4 | $dx=1$ sposta la lettura di una colonna verso destra; l'ultima colonna di `alin` rimane senza corrispondenza <br> (diventa 0) e viene sistematicamente marcata; anche il difetto reale (90) viene rilevato. |
| 2<br>2<br>0 0<br>20<br>10 10<br>10 10<br>10 10<br>10 60 | 10 10<br>10 60<br>0 0<br>0 1<br>Totale di pixel difettosi: 1 | Senza spostamento ($dx=dy=0$): `alin` è identica a `cap`; solo il difetto reale (60) viene rilevato. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0607" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0607 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0607 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0607 button:hover { background: #e8dfcf; }
  #sim-ep0607 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0607_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0607_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0607_cell { width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 10px; font-weight: 700; font-family: monospace; border: 1px solid #e4dcc8; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP06_07: Registro a Traslazione + Sottrazione</span>
  <span class="sim-ep0607_pill">|ref &minus; alin(dx,dy)| &gt; T</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0607_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Spostamento Orizzontale (dx): <span id="sim-ep0607_vl_dx" style="font-family:monospace; color:#26241d;">1</span>
      </label>
    </div>
    <input id="sim-ep0607_sl_dx" type="range" min="-2" max="2" step="1" value="1">

    <div style="display:flex; justify-content:space-between; align-items:center; margin:10px 0 4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Soglia (T): <span id="sim-ep0607_vl_t" style="font-family:monospace; color:#26241d;">30</span>
      </label>
    </div>
    <input id="sim-ep0607_sl_t" type="range" min="0" max="100" step="5" value="30">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Regola lo spostamento del nastro (dx) e la soglia T. Osserva come il bordo "fantasma" scompare quando dx = 0.
    </div>
  </div>

  <!-- Exibição das Grades 3x3 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(160px, 1fr)); gap:12px; margin-bottom:14px;">
    
    <div class="sim-ep0607_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        rif
      </div>
      <div id="sim-ep0607_g_ref" style="display:grid; grid-template-columns:repeat(3, 40px); gap:3px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0607_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        alin (registrato)
      </div>
      <div id="sim-ep0607_g_alin" style="display:grid; grid-template-columns:repeat(3, 40px); gap:3px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0607_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        maschera
      </div>
      <div id="sim-ep0607_g_mask" style="display:grid; grid-template-columns:repeat(3, 40px); gap:3px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0607_debug" class="sim-ep0607_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep07(root){
    if (!root || root.dataset.sim06Ep07Init) return;
    root.dataset.sim06Ep07Init = "1";

    var N = 3;
    var ref = [[50, 50, 50], [50, 50, 50], [50, 50, 50]];
    // cap representa a peça deslocada 1 px à direita (col 0 = 0) mais um defeito em (1,2)
    var cap = [[0, 50, 50], [0, 50, 90], [0, 50, 50]];

    var slDx  = root.querySelector('#sim-ep0607_sl_dx');
    var vlDx  = root.querySelector('#sim-ep0607_vl_dx');
    var slT   = root.querySelector('#sim-ep0607_sl_t');
    var vlT   = root.querySelector('#sim-ep0607_vl_t');
    var gRef  = root.querySelector('#sim-ep0607_g_ref');
    var gAlin = root.querySelector('#sim-ep0607_g_alin');
    var gMask = root.querySelector('#sim-ep0607_g_mask');
    var dbg   = root.querySelector('#sim-ep0607_debug');

    function cellStyle(g){
      var v = Math.max(0, Math.min(255, g));
      return 'background:rgb(' + v + ',' + v + ',' + v + '); color:' + (v > 140 ? '#000000' : '#ffffff') + ';';
    }

    function render(){
      var dx = parseInt(slDx.value, 10);
      var T  = parseInt(slT.value, 10);
      vlDx.textContent = dx;
      vlT.textContent  = T;

      gRef.innerHTML  = '';
      gAlin.innerHTML = '';
      gMask.innerHTML = '';

      var alin = [], mask = [], total = 0;

      for (var i = 0; i < N; i++){
        var rowA = [], rowM = [];
        for (var j = 0; j < N; j++){
          var pj = j + dx;
          var v = (pj >= 0 && pj < N) ? cap[i][pj] : 0;
          rowA.push(v);

          var diff = Math.abs(ref[i][j] - v);
          var m = diff > T ? 1 : 0;
          if (m) total++;
          rowM.push(m);
        }
        alin.push(rowA);
        mask.push(rowM);
      }

      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var cr = document.createElement('div');
          cr.className = 'sim-ep0607_cell';
          cr.style.cssText = cellStyle(ref[i][j]);
          cr.textContent = ref[i][j];
          gRef.appendChild(cr);

          var ca = document.createElement('div');
          ca.className = 'sim-ep0607_cell';
          ca.style.cssText = cellStyle(alin[i][j]);
          ca.textContent = alin[i][j];
          gAlin.appendChild(ca);

          var m = mask[i][j];
          var cm = document.createElement('div');
          cm.className = 'sim-ep0607_cell';
          if (m) {
            cm.style.cssText = 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
          } else {
            cm.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
          }
          cm.textContent = m;
          gMask.appendChild(cm);
        }
      }

      if (total > 0) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      }

      dbg.textContent = 'dx = ' + dx + '  |  T = ' + T + '  |  Total de pixels defeituosos: ' + total + ' / 9';
    }

    slDx.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep07(){
    var root = document.getElementById('sim-ep0607');
    if (root) initSim06Ep07(root); else setTimeout(tryInitSim06Ep07, 200);
  }
  tryInitSim06Ep07();
})();
</script>
""")

**Figura 6.7:** Simulatore EP06_07: *Pipeline* di Ispezione — Registrazione per Traslazione e Sottrazione


<figure id="fig-06-sim-ep0607">
  <img src="imagens/fig-06-sim-ep0607.png" alt=" Simulatore EP06_07: *Pipeline* di Ispezione — Registrazione per Traslazione e Sottrazione " style="max-width:80%" />
  <figcaption><strong>Figura 6.7:</strong>  Simulatore EP06_07: *Pipeline* di Ispezione — Registrazione per Traslazione e Sottrazione </figcaption>
</figure>

In [ ]:
%%writefile EP06_07.py
# Codice Python

In [ ]:
TestSuite("EP06_07.py").run()

### EP06_08 ⚫ Segmentazione e Decodifica Reale di *QRCode* con OpenCV

Negli esercizi precedenti, le grandezze intermedie del *pipeline* di elaborazione delle immagini — come aree, perimetri, varianze e spostamenti — sono state fornite direttamente o calcolate a partire da matrici numeriche, senza la necessità di librerie specializzate di Visione Artificiale. In questo esercizio conclusivo del capitolo, questa restrizione viene rimossa intenzionalmente: verrà utilizzata la libreria **OpenCV** (`cv2`) per localizzare e decodificare un *QRCode* reale presente in una scena.

La proposta riproduce un flusso semplificato di sistemi impiegati nell'ispezione visiva, nell'automazione industriale e nella lettura automatica di documenti. Per mantenere l'input dei dati accessibile al contesto educativo, il caricamento dell'immagine sarà integrato nella libreria didattica `morph`, tramite la funzione `mm.readImg`.

La scena è fornita nel formato **PGM ASCII (P2)** e contiene un singolo *QRCode* valido, oltre a diversi **oggetti distrattori**, come rettangoli, regioni di rumore texturizzato e blocchi isolati. La segmentazione basata esclusivamente su proprietà geometriche — come area e forma approssimativamente quadrata — è necessaria per ridurre lo spazio di ricerca, ma non è sufficiente per identificare il codice corretto. La conferma finale sarà effettuata esclusivamente tramite il tentativo di decodifica utilizzando `cv2.QRCodeDetector`, una procedura compatibile con applicazioni reali di riconoscimento automatico.

#### 📋 Linee Guida di Implementazione

1. **Lettura delle dimensioni e dei parametri**

   Leggere, in questo ordine, tramite l'input standard:

   - una riga contenente il numero di righe $L$;
   - una riga contenente il numero di colonne $C$;
   - una riga contenente i quattro parametri dell'algoritmo separati da spazio:
     - soglia di binarizzazione $T$ (intero);
     - area minima $A_{\text{min}}$ (intero);
     - tolleranza di aspetto $\text{tol}$ (reale);
     - margine $M$ (intero, in pixel).

2. **Caricamento dell'immagine**

   Utilizzare la funzione didattica `f = mm.readImg(L, C)` per leggere i valori $L \times C$ dell'immagine in scala di grigi, ottenendo un *array* NumPy di tipo `uint8`.

3. **Binarizzazione**

   Applicare una sogliatura binaria invertita utilizzando la soglia $T$. Ogni pixel dell'immagine originale con intensità strettamente maggiore di $T$ deve essere convertito a 255, mentre i rimanenti devono assumere il valore 0.

4. **Rilevamento dei contorni**

   Estrarre i componenti connessi esterni utilizzando `cv2.findContours(...)` con i parametri:

   * `cv2.RETR_EXTERNAL`;
   * `cv2.CHAIN_APPROX_SIMPLE`.

5. **Filtraggio geometrico**

   Per ogni contorno trovato:

   * calcolare il rettangolo delimitatore `(x, y, w, h)` tramite `cv2.boundingRect`;
   * mantenere solo i candidati che soddisfano simultaneamente:

     **Area minima**

     $$
     w \times h > A_{\text{min}}
     $$

     **Rapporto d'aspetto**

     $$
     \left|\frac{w}{h}-1\right| \le \text{tol}
     $$

6. **Ordinamento dei candidati**

   Ordinare i candidati per area del rettangolo delimitatore

   $$
   w \times h
   $$

   in ordine decrescente.

   In caso di parità, preservare l'ordine originariamente restituito da `cv2.findContours`.

7. **Verifica tramite decodifica**

   Per ogni candidato, seguendo l'ordine stabilito:

   * espandere il rettangolo di $M$ pixel nelle quattro direzioni;
   * limitare gli indici per rimanere all'interno dell'immagine;
   * estrarre il ritaglio direttamente dall'immagine originale `f`;
   * applicare `cv2.QRCodeDetector().detectAndDecode(...)` su tale ritaglio.

8. **Criterio di arresto**

   Interrompere immediatamente l'elaborazione quando il primo candidato produce una *stringa* decodificata non vuota.

9. **Caso non trovato**

   Se nessun candidato viene decodificato con successo, stampare esattamente: `QRCODE_NAO_ENCONTRATO`

10. **Output (caso trovato)**

    Stampare due righe.

    Prima riga: `riga colonna altezza larghezza` utilizzando il rettangolo delimitatore **originale**, prima dell'espansione tramite il margine $M$.

    Seconda riga: `testo_decodificato`

#### 📌 Vincoli Computazionali

* Utilizzare funzioni OpenCV per eseguire la binarizzazione, il rilevamento dei contorni, il calcolo del rettangolo delimitatore e la decodifica del QRCode.
* Il filtraggio geometrico deve avvenire obbligatoriamente prima della fase di decodifica.
* Utilizzare esclusivamente la soglia fissa $T$ fornita in input. Non è consentito utilizzare metodi automatici di sogliatura, come Otsu o sogliatura adattativa.
* Garantire che i ritagli inviati al decodificatore rimangano all'interno dei limiti dell'immagine.

#### 🧠 Fondamenti Teorici

| Fase                     | Ruolo nel *pipeline*                                                                                      | Conseguenza se omessa                                                                      |
| ------------------------ | -------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------ |
| **Filtraggio geometrico** | Riduce lo spazio di ricerca selezionando solo regioni compatibili con la geometria attesa di un QRCode.  | Il decodificatore elaborerebbe tutti i contorni, inclusi rumori e oggetti distrattori.     |
| **Decodifica**           | Conferma semanticamente se il candidato contiene un QRCode valido.                                       | Oggetti geometricamente simili potrebbero essere classificati erroneamente come QRCode.    |
| **Margine $M$**          | Preserva la *zona di silenzio* attorno al codice, facilitandone il rilevamento.                          | L'assenza di tale margine può impedire l'allineamento e la corretta lettura del codice.    |

Questo esercizio integra concetti studiati durante il capitolo in un unico *pipeline* di Visione Artificiale. La segmentazione riduce l'insieme delle regioni candidate tramite caratteristiche geometriche, mentre la fase di decodifica valida il contenuto della regione utilizzando un algoritmo specializzato di riconoscimento.

#### 📦 Specifica di Input e Output (VPL)

**Struttura di Input**

```
L
C
T A_min tol M
[matrice dell'immagine]
```

**Struttura di Output (Successo)**

```
riga colonna altezza larghezza
testo_decodificato
```

**Struttura di Output (Fallimento)**

```
QRCODE_NAO_ENCONTRATO
```

#### 📌 File di Riferimento (.pgm)

A scopo di validazione, debug locale e analisi di matrici reali di pixel, i file immagine generati nel formato ASCII P2 sono disponibili nella directory del progetto. Puoi utilizzarli per testare con i decoder del tuo cellulare l'aderenza del tuo codice (salvare i *.pgm localmente per visualizzarli):

* 📥 **[Caso 1: Pattern Normale](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso1_Normal.pgm)** – Contiene un singolo codice perfettamente centralizzato con semplici distrattori geometrici alla periferia.
* 📥 **[Caso 2: Scenario Complesso](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso2_Complexo.pgm)** – Presenta una maggiore densità di rumore texturizzato e molteplici candidati distrattori che mettono alla prova i limiti del filtraggio per aspetto.
* 📥 **[Caso 3: Messaggio Esteso](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso3_MensagemSecreta.pgm)** – Contiene un QRCode strutturato a partire da una stringa di caratteri di lunghezza maggiore, generando una maggiore densità di moduli interni.
* 📥 **[Caso 4: Geometria Compatta](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso4_Excelente.pgm)** – Valuta il comportamento del *pipeline* in condizioni ottimizzate di contrasto e posizionamento limite.
* 📥 **[Caso 5: Scenario di Esclusione](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso5_Nao_Encontrado.pgm)** – Immagine composta puramente da elementi distrattori di alta area, progettata per validare il comportamento di fallimento controllato del programma.

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0608" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<!-- Cabeçalho no padrão institucional -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">📋 Simulatore EP06_08: Segmentazione e Decodifica di QRCode</span>
  <span style="font-size:10px;font-weight:700;padding:3px 10px;border-radius:40px;border:1px solid #e4dcc8;background:#26241d;color:#7ee7c6;font-family:monospace;">Filtro Geometrico &rarr; Criterio di Arresto Semantico</span>
</div>

<div style="padding:16px;background:#ffffff;">
  <p style="margin:0 0 14px 0;font-size:11px;color:#8a8371;line-height:1.5;text-align:center;font-weight:600;">
    Regola interattivamente i parametri di ingresso dell'algoritmo (A_min e tol) per verificare quali componenti vengono filtrati geometricamente e come il criterio di arresto tramite analisi semantica interrompe la scansione della coda.
  </p>
  
  <div style="display:flex;gap:14px;margin-bottom:14px;flex-wrap:wrap;">
    <!-- Slider Area Minima -->
    <div style="flex:1;min-width:200px;background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#5e5a4a;">Area minima (A_min, px&sup2;)</label>
        <span id="sim-ep0608_vl_amin" style="font-family:monospace;font-weight:700;color:#26241d;">250</span>
      </div>
      <input id="sim-ep0608_sl_amin" style="width:100%;accent-color:#26241d;cursor:pointer;height:4px;" max="3000" min="0" step="50" type="range" value="250">
    </div>
    
    <!-- Slider Tolerancia -->
    <div style="flex:1;min-width:200px;background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#5e5a4a;">Tolleranza di aspetto (tol)</label>
        <span id="sim-ep0608_vl_tol" style="font-family:monospace;font-weight:700;color:#26241d;">0.22</span>
      </div>
      <input id="sim-ep0608_sl_tol" style="width:100%;accent-color:#26241d;cursor:pointer;height:4px;" max="1.0" min="0.05" step="0.01" type="range" value="0.22">
    </div>
  </div>

  <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(240px, 1fr));gap:14px;margin-bottom:14px;">
    <!-- Canvas da Cena -->
    <div style="text-align:center;background:#fafaf7;border:1px solid #e9e3d3;padding:14px;border-radius:12px;">
      <div style="font-size:10px;font-weight:700;color:#8a8371;text-transform:uppercase;margin-bottom:10px;letter-spacing:0.04em;">Visualizzazione della Scena (Matrice f)</div>
      <canvas id="sim-ep0608_canvas" width="260" height="260" style="border:1px solid #e4dcc8;border-radius:10px;background:#ffffff;margin:0 auto;display:block;"></canvas>
    </div>
    
    <!-- Lista de Candidatos -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;padding:14px;border-radius:12px;">
      <div style="font-size:10px;font-weight:700;color:#8a8371;text-transform:uppercase;margin-bottom:10px;text-align:center;letter-spacing:0.04em;">Componenti Connessi nella Coda</div>
      <div id="sim-ep0608_lista" style="font-family:monospace;font-size:11px;display:flex;flex-direction:column;gap:8px;"></div>
    </div>
  </div>
  
  <!-- Console de Saída VPL -->
  <div id="sim-ep0608_debug" style="background:#fafaf7;border-radius:12px;padding:12px;border:1px solid #e9e3d3;font-family:monospace;font-size:11px;color:#26241d;text-align:center;"></div>
</div>

<script>
(function(){
  function initSim06Ep08(root){
    if(!root || root.dataset.sim06Ep08Init) return;
    root.dataset.sim06Ep08Init = "1";

    var formas = [
      {x: 145, y: 35,  w: 76, h: 76, tipo: "Componente QRCode Real", cor: "#cbd5e1", decodifica: true, padrao: "qr"},
      {x: 35,  y: 145, w: 55, h: 68, tipo: "Falso QRCode (Assimétrico)", cor: "#e2e8f0", decodifica: false, padrao: "falso_qr"},
      {x: 45,  y: 35,  w: 44, h: 44, tipo: "Círculo / Distrator", cor: "#f1f5f9", decodifica: false, padrao: "circulo"},
      {x: 160, y: 175, w: 68, h: 26, tipo: "Retângulo Distrator", cor: "#e2e8f0", decodifica: false, padrao: "retangulo"},
      {x: 65,  y: 220, w: 14, h: 14, tipo: "Ruído Isolado", cor: "#f8fafc", decodifica: false, padrao: "ruido"}
    ];

    var slA = root.querySelector('#sim-ep0608_sl_amin');
    var vlA = root.querySelector('#sim-ep0608_vl_amin');
    var slT = root.querySelector('#sim-ep0608_sl_tol');
    var vlT = root.querySelector('#sim-ep0608_vl_tol');
    var canvas = root.querySelector('#sim-ep0608_canvas');
    var ctx = canvas.getContext('2d');
    var lista = root.querySelector('#sim-ep0608_lista');
    var dbg = root.querySelector('#sim-ep0608_debug');

    function desenhaForma(f, estado){
      ctx.save();
      
      var corBorda = '#94a3b8';
      if (estado === 'candidato_ok') corBorda = '#10b981';
      if (estado === 'candidato_falhou') corBorda = '#f43f5e';
      if (estado === 'rejeitado') corBorda = '#cbd5e1';

      ctx.lineWidth = (estado === 'candidato_ok' || estado === 'candidato_falhou') ? 3 : 1.5;
      ctx.strokeStyle = corBorda;

      if (estado === 'rejeitado') {
        ctx.fillStyle = '#f8fafc';
      } else {
        if(f.padrao === 'qr') ctx.fillStyle = '#e2e8f0';
        else if(f.padrao === 'retangulo') ctx.fillStyle = '#fffbeb';
        else if(f.padrao === 'falso_qr') ctx.fillStyle = '#f0f9ff';
        else ctx.fillStyle = '#fdf4ff';
      }

      if(f.padrao === 'circulo'){
        ctx.beginPath();
        ctx.arc(f.x + f.w/2, f.y + f.h/2, f.w/2, 0, 2 * Math.PI);
        ctx.fill(); ctx.stroke();
      } else {
        ctx.fillRect(f.x, f.y, f.w, f.h);
        ctx.strokeRect(f.x, f.y, f.w, f.h);
        
        if(f.padrao === 'qr' || f.padrao === 'falso_qr'){
          var c = f.w / 5;
          ctx.fillStyle = '#ffffff';
          [[f.x + 3, f.y + 3], [f.x + f.w - c - 3, f.y + 3], [f.x + 3, f.y + f.h - c - 3]].forEach(function(p){
            ctx.fillRect(p[0], p[1], c, c);
            ctx.strokeRect(p[0], p[1], c, c);
          });
          
          ctx.fillStyle = (f.padrao === 'qr') ? '#334155' : '#64748b';
          [[f.x + 5, f.y + 5], [f.x + f.w - c + 1, f.y + 5], [f.x + 5, f.y + f.h - c + 1]].forEach(function(p){
            ctx.fillRect(p[0], p[1], c - 4, c - 4);
          });
        }
      }
      ctx.restore();
    }

    function render(){
      var amin = parseInt(slA.value, 10);
      var tol = parseFloat(slT.value);
      vlA.textContent = amin;
      vlT.textContent = tol.toFixed(2);

      ctx.clearRect(0, 0, canvas.width, canvas.height);
      ctx.fillStyle = '#ffffff';
      ctx.fillRect(0, 0, canvas.width, canvas.height);

      var candidatos = formas.map(function(f){
        var area = f.w * f.h;
        var aspecto = f.w / f.h;
        var passaArea = area > amin;
        var passaAspecto = Math.abs(aspecto - 1.0) <= tol;
        return {f: f, area: area, aspecto: aspecto, passa: passaArea && passaAspecto};
      }).sort(function(a, b){ return b.area - a.area; });

      lista.innerHTML = '';
      var encontrado = null;
      var flagParada = false;

      candidatos.forEach(function(c){
        var estado, texto, bgBox, txBox;
        
        if(!c.passa){
          estado = 'rejeitado';
          texto = 'REJEITADO (Área = ' + c.area + ' px&sup2;, Aspeto = ' + c.aspecto.toFixed(2) + ')';
          bgBox = '#f1f5f9';
          txBox = '#94a3b8';
        } else if(flagParada){
          estado = 'rejeitado';
          texto = 'FILA INTERROMPIDA (Critério de Parada Ativo)';
          bgBox = '#f8fafc';
          txBox = '#cbd5e1';
        } else if(c.f.decodifica){
          estado = 'candidato_ok';
          texto = 'SUCESSO: DECODIFICADO &#10004;';
          bgBox = '#ecfdf5';
          txBox = '#059669';
          encontrado = c.f;
          flagParada = true;
        } else {
          estado = 'candidato_falhou';
          texto = 'GEOMETRIA OK &rarr; FALHA NA DECODIFICAÇÃO &#10008;';
          bgBox = '#fff5f5';
          txBox = '#e11d48';
        }
        
        desenhaForma(c.f, estado);
        
        var div = document.createElement('div');
        div.style.cssText = 'padding:8px 10px;border-radius:8px;background:' + bgBox + ';border:1px solid #edf2f7;color:' + txBox + ';display:flex;flex-direction:column;gap:2px;';
        
        var nameSpan = document.createElement('strong');
        nameSpan.style.fontSize = '11px';
        nameSpan.textContent = c.f.tipo + ' (' + c.area + ' px²)';
        
        var statusSpan = document.createElement('span');
        statusSpan.style.fontSize = '10px';
        statusSpan.style.opacity = '0.9';
        statusSpan.innerHTML = texto;

        div.appendChild(nameSpan);
        div.appendChild(statusSpan);
        lista.appendChild(div);
      });

      if (encontrado) {
        dbg.style.backgroundColor = '#eafaf1';
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.color = '#04342C';
        dbg.innerHTML = '<div style="text-align:left;font-weight:700;color:#04342C;margin-bottom:4px;">&#128994; SAÍDA PADRÃO (VPL):</div>' +
                        'y=' + encontrado.y + ' x=' + encontrado.x + ' h=' + encontrado.h + ' w=' + encontrado.w + '<br>' +
                        '<span style="color:#04342C;font-weight:700;">"EP06_08 - PDI-VC | Parabens! Voce decodificou este QR Code!"</span>';
      } else {
        dbg.style.backgroundColor = '#fdecea';
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.color = '#c0392b';
        dbg.innerHTML = '<div style="text-align:left;font-weight:700;color:#c0392b;margin-bottom:4px;">&#128308; SAÍDA PADRÃO (VPL):</div>' +
                        'QRCODE_NAO_ENCONTRADO';
      }
    }

    slA.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }
  
  function tryInitSim06Ep08(){
    var root = document.getElementById('sim-ep0608');
    if(root) initSim06Ep08(root); else setTimeout(tryInitSim06Ep08, 200);
  }
  tryInitSim06Ep08();
})();
</script>
</div>
""")

**Figura 6.8:** Simulatore EP06_08: Segmentazione Geometrica + Verifica tramite Decodifica di QRCode


<figure id="fig-06-sim-ep0608">
  <img src="imagens/fig-06-sim-ep0608.png" alt=" Simulatore EP06_08: Segmentazione Geometrica + Verifica tramite Decodifica di QRCode " style="max-width:80%" />
  <figcaption><strong>Figura 6.8:</strong>  Simulatore EP06_08: Segmentazione Geometrica + Verifica tramite Decodifica di QRCode </figcaption>
</figure>

In [ ]:
%%writefile EP06_08.py
# Codice Python

In [ ]:
TestSuite("EP06_08.py").run()